# Flujo compresible adiabático entre reservorios (régimen subsónico)

En este notebook se resuelven problemas de **cálculo de flujo másico** en una tubería de sección
constante que conecta dos reservorios de gas:

- Reservorio aguas arriba a presión absoluta \(P_0\) y temperatura \(T_0\).
- Reservorio aguas abajo a presión absoluta \(P_2\).
- Flujo estacionario, **adiabático**, **unidimensional**, sin trabajo de eje y con área constante.
- El gas se modela como **gas ideal** con propiedades constantes \((R, \gamma)\).
- La conducción se modela como un ducto con fricción distribuida caracterizada por un
  **factor de fricción de Fanning** \(f\) (indirectamente dado a partir del \(f_T\) de Darcy de las tablas de Crane),
  y una **longitud equivalente** \(L_e\).

El objetivo es, dados \(P_0, P_2, T_0, D, L_e, f_T\), calcular el **flujo másico** aproximado \(\dot m\)
bajo el modelo de flujo adiabático subsónico entre reservorios.


## Modelo físico y ecuaciones reducidas

Tomamos como estado de referencia el reservorio aguas arriba (tanque A):

- Presión absoluta: \(P_0\).
- Temperatura: \(T_0\).
- Volumen específico (gas ideal): 
  \[
  v_0 = \frac{R T_0}{P_0}.
  \]

El gas es ideal con:
- Constante de los gases: \(R\).
- Relación de calores específicos: \(\gamma = c_p/c_v\).

Definimos las variables adimensionales:
- \[
  r = \frac{v_0}{v_{1'}}, \qquad
  z = \frac{v_0}{v_2},
  \]
  donde \(v_{1'}\) es el volumen específico en la entrada efectiva de la tubería
  y \(v_2\) el volumen específico en la salida hacia el segundo reservorio.
- \[
  F = \frac{P_2}{P_0}
  \]
  es la relación de presiones entre reservorios.
- \[
  X = \left(\frac{\dot m}{A}\right)^2 \frac{v_0}{P_0},
  \]
  donde \(A\) es el área interna de la tubería.

Bajo las hipótesis de flujo adiabático subsónico en ducto de sección constante, el sistema reducido es:

1. **Relación entre entrada desde reservorio y salida**:
   \[
   \frac{\gamma-1}{2\gamma}\, X = r^2 - r^{\gamma+1}
   \]
   \[
   \frac{\gamma-1}{2\gamma}\, X = z^2 - F z.
   \]

2. **Definición de \(X\)** (la usaremos para eliminar \(X\)):
   \[
   X = C\,(z^2 - F z), \qquad C = \frac{2\gamma}{\gamma-1}.
   \]

3. **Ecuación de Fanno reducida (entre \(1'\) y \(2\))**:
   \[
   \frac{r^2 - z^2}{X} - \frac{\gamma+1}{\gamma}\ln\!\left(\frac{r}{z}\right)
   = f_{\text{Fanning}} \frac{L_e}{D}.
   \]

En la práctica, las tablas de Crane dan un **factor de Darcy** \(f_T\). Para un flujo completamente turbulento:
\[
f_{\text{Darcy}} = f_T, \qquad f_{\text{Darcy}} = 4 f_{\text{Fanning}}
\quad\Rightarrow\quad
f_{\text{Fanning}} = \frac{f_T}{4}.
\]

Al sustituir, el término de la derecha se puede escribir directamente como
\[
\text{RHS} = f_T\,\frac{L_e}{D},
\]
cuando las ecuaciones están escritas originalmente en términos de \(f_{\text{Fanning}}\) y se ha absorbido el factor 4 en la definición efectiva de \(f_T\).

En este notebook usaremos explícitamente:
\[
E_1(r,z) = r^2 - r^{\gamma+1} - z^2 + F z = 0,
\]
\[
E_2(r,z) =
\frac{r^{2}-z^{2}}{X(r,z)}
- \frac{\gamma+1}{\gamma}\ln\!\left(\frac{r}{z}\right)
- f_T \frac{L_e}{D} = 0,
\]
con
\[
X(r,z) = C\,(z^2 - F z), \qquad C = \frac{2\gamma}{\gamma-1}.
\]

Una vez hallados \(r\) y \(z\), el flujo másico se obtiene de
\[
\dot m = A\,\sqrt{\frac{X P_0}{v_0}}.
\]


In [1]:
### Paquetes necesarios

using NonlinearSolve


## Funciones auxiliares: sistema reducido y cálculo de \(\dot m\)

Definimos:

- `residuos_rz!` que implementa \(E_1(r,z)\) y \(E_2(r,z)\).
- `resolver_rz` que, dado un conjunto de parámetros, resuelve el sistema
  en \(r\) y \(z\) usando `NonlinearSolve.jl`.
- `flujo_masico_reservorios_subsonico` que devuelve
  \(\dot m\) y las variables adimensionales.


In [ ]:
struct ParametrosFlujoReservorio
    γ::Float64
    F::Float64
    ft::Float64   # factor de Darcy tipo Crane
    Leq::Float64  # longitud equivalente [m]
    D::Float64    # diámetro interno [m]
    P0::Float64   # presión absoluta del reservorio aguas arriba [Pa]
    T0::Float64   # temperatura absoluta del reservorio [K]
    Rgas::Float64 # constante de los gases [J/kg·K]
end

function residuos_rz!(Fvec, u, p::ParametrosFlujoReservorio)
    r = u[1]
    z = u[2]

    γ   = p.γ
    Fp  = p.F
    ft  = p.ft
    Leq = p.Leq
    D   = p.D

    C = 2γ/(γ - 1)

    if !(0.0 < z < r < 1.0)
        Fvec[1] = 1e6
        Fvec[2] = 1e6
        return
    end

    B = z^2 - Fp*z
    if B <= 0.0
        Fvec[1] = 1e6
        Fvec[2] = 1e6
        return
    end

    X = C * B
    N = r^2 - z^2

    Fvec[1] = r^2 - r^(γ + 1) - z^2 + Fp*z

    RHS = ft * Leq / D
    Fvec[2] = N/X - (γ + 1)/γ * log(r/z) - RHS
end

function resolver_rz(p::ParametrosFlujoReservorio;
                     r0::Float64 = 0.97,
                     z0::Float64 = p.F + 0.01)
    u0 = [r0, z0]
    prob = NonlinearProblem(residuos_rz!, u0, p)
    sol  = solve(prob; abstol = 1e-15, reltol = 1e-15, maxiters = 500)
    return sol
end

function flujo_masico_reservorios_subsonico(p::ParametrosFlujoReservorio)
    sol = resolver_rz(p)
    r, z = sol.u

    γ   = p.γ
    Fp  = p.F
    P0  = p.P0
    T0  = p.T0
    Rg  = p.Rgas
    D   = p.D

    v0 = Rg * T0 / P0

    C = 2γ/(γ - 1)
    X = C * (z^2 - Fp*z)

    A = π * D^2 / 4

    mdot = A * sqrt(X * P0 / v0)

    return (; r, z, X, mdot, sol)
end


flujo_masico_reservorios_subsonico (generic function with 1 method)

## Ejemplo 1: dos tanques de nitrógeno conectados por una tubería

Dos tanques de volumen \(10\ \mathrm{m^3}\) contienen nitrógeno y están conectados por una tubería de
acero comercial Sch 40 de \(1/2"\) de diámetro nominal, longitud equivalente \(L_e = 18.5\ \mathrm{m}\).

- Presión tanque A (aguas arriba): \(P_A = 8.0\ \text{bar(g)} \Rightarrow P_0 = 9.0\ \text{bar(abs)}\).
- Presión tanque B (aguas abajo): \(P_B = 4.0\ \text{bar(g)} \Rightarrow P_2 = 5.0\ \text{bar(abs)}\).
- Temperatura en ambos tanques: \(T \approx 288\ \mathrm{K}\).
- Nitrógeno:
  - \(M = 28\ \text{g/mol}\),
  - \(R = R_u/M \approx 8.314/0.028\ \mathrm{J/kg·K}\),
  - \(\gamma \approx 1.4\).
- Factor de fricción de Darcy tomado de Crane para tubo 1/2" en régimen turbulento completamente desarrollado:
  \[
  f_T = 0.026.
  \]
- Diámetro interno aproximado de 1/2" Sch 40:
  \[
  D \approx 0.0158\ \mathrm{m}.
  \]

La relación de presiones es:
\[
F = \frac{P_2}{P_0} = \frac{5}{9}.
\]

Queremos estimar el **flujo másico inicial** \(\dot m\) bajo el modelo de flujo adiabático subsónico
entre los reservorios.


In [3]:
γ   = 1.4
P0  = 9e5        # Pa
P2  = 5e5        # Pa
F   = P2 / P0

T0   = 288.0      # K
Rgas = 8.314 / 0.028  # J/(kg·K) N2 aprox.

ft  = 0.026      # Darcy (Crane)
Leq = 18.5       # m
D   = 0.0158     # m

p = ParametrosFlujoReservorio(γ, F, ft, Leq, D, P0, T0, Rgas)

resultado = flujo_masico_reservorios_subsonico(p)

@show resultado.r;
@show resultado.z;
@show resultado.X;
@show resultado.mdot;


resultado.r = 0.9922843799308645
resultado.z = 0.5609850586197684
resultado.X = 0.02132109066327559
resultado.mdot = 0.08811077777374389


## Ejercicio: experimentar con otros casos

Puede modificarse:

- La relación de presiones \(F = P_2/P_0\).
- La longitud equivalente \(L_e\).
- El diámetro \(D\).
- El factor de fricción \(f_T\).

para estudiar el efecto de cada parámetro sobre el flujo másico \(\dot m\).


In [4]:
γ_nuevo   = 1.4
P0_nuevo  = 9e5
P2_nuevo  = 6e5
F_nuevo   = P2_nuevo / P0_nuevo

T0_nuevo   = 288.0
Rgas_nuevo = 8.314 / 0.028

ft_nuevo  = 0.026
Leq_nuevo = 10.0
D_nuevo   = 0.020

p_nuevo = ParametrosFlujoReservorio(γ_nuevo, F_nuevo, ft_nuevo, Leq_nuevo, D_nuevo,
                                    P0_nuevo, T0_nuevo, Rgas_nuevo)

resultado_nuevo = flujo_masico_reservorios_subsonico(p_nuevo)

@show resultado_nuevo.r;
@show resultado_nuevo.z;
@show resultado_nuevo.X;
@show resultado_nuevo.mdot;


resultado_nuevo.r = 0.9861416701066252
resultado_nuevo.z = 0.6746901030754701
resultado_nuevo.X = 0.037893331963725414
resultado_nuevo.mdot = 0.18821383379583123


## Extensión: interfaz de alto nivel para datos en unidades habituales

En esta sección se define una interfaz más "de ingeniería de procesos", que permite trabajar
directamente con:

- Presiones en bar absolutos \(P_{0,\text{bar}}, P_{2,\text{bar}}\).
- Temperatura en °C.
- Diámetro nominal de la tubería (por ejemplo `"1/2"` para 1/2" Sch 40).
- Selección del gas mediante una etiqueta (`"N2"`, `"air"`, etc.).

La idea es construir automáticamente la estructura `ParametrosFlujoReservorio` a partir de estos
datos de entrada y de **valores típicos** de \(\gamma\) y \(R\) para el gas, y de un valor tabulado
de \(f_T\) para el diámetro nominal (cuando esté disponible).


In [12]:
### Datos típicos de gases y factores de fricción tabulados

struct DatosGas
    γ::Float64
    R::Float64
end

const GASES = Dict(
    "N2"  => DatosGas(1.40, 8.314/0.02800),
    "air" => DatosGas(1.40, 8.314/0.02897),
    "CO2" => DatosGas(1.30, 8.314/0.04401),
)

"""
Datos geométricos e hidráulicos de un tubo para un diámetro nominal dado.
- Dint: diámetro interno [m]
- ft: factor de fricción Darcy (Crane)
"""
struct DatosTubo
    Dint::Float64   # diámetro interno [m]
    ft::Float64     # factor de fricción Darcy f_T
end

"""
Diccionario de diámetros nominales (DN) a datos de tubo (Dint, f_T).
COMPLETAR con los valores reales de tu tabla Crane/Sch40.
"""
const TUBOS_SCH40 = Dict(
    # EJEMPLOS: reemplazar por tus datos reales
    "1/2"   => DatosTubo(0.0158, 0.027), # acá el dint ya esta en m
    "3/4"   => DatosTubo(0.824, 0.025),  # acá el dint esta en "
    "1"     => DatosTubo(0.0266, 0.023), # falta Dint en "
    "1 1/4" => DatosTubo(0.0350, 0.022), # falta Dint en "
    "1 1/2" => DatosTubo(0.0409, 0.021), # falta Dint en "
    "2"     => DatosTubo(0.0525, 0.019), # falta Dint en "
)

TUBOS_SCH40

In [13]:
"""
Crea un objeto ParametrosFlujoReservorio a partir de datos en unidades habituales:
- P0_bar, P2_bar: presiones absolutas en bar.
- T0_C: temperatura en °C.
- DN: diámetro nominal (por ejemplo "1/2" para 1/2" Sch 40).
- gas: etiqueta del gas ("N2", "air", "CO2", ...).
- Leq: longitud equivalente [m].
- ft: si es `nothing`, se toma el valor tabulado de TUBOS_SCH40;
      si se da un Float64, se usa ese valor y se ignora el tabulado.
"""
function parametros_desde_entrada(P0_bar::Float64,
                                  P2_bar::Float64,
                                  T0_C::Float64,
                                  DN::AbstractString;
                                  gas::AbstractString = "N2",
                                  Leq::Float64 = 10.0,
                                  ft::Union{Nothing,Float64} = nothing)

    @assert P2_bar < P0_bar "Se requiere P2_bar < P0_bar para flujo de A hacia B."
    @assert haskey(GASES, gas) "Gas '$gas' no definido en el diccionario GASES."
    @assert haskey(TUBOS_SCH40, DN) "DN = $DN no definido en TUBOS_SCH40."

    gasdata = GASES[gas]
    γ    = gasdata.γ
    Rgas = gasdata.R

    # Conversión de unidades
    P0 = P0_bar * 1e5
    P2 = P2_bar * 1e5
    F  = P2 / P0

    T0 = T0_C + 273.15

    # Datos de la línea (Dint y f_T) desde el diccionario
    tubo = TUBOS_SCH40[DN]
    D    = tubo.Dint
    ft_tab = tubo.ft

    # Si el usuario pasa ft explícito, pisa el tabulado
    ft_val = ft === nothing ? ft_tab : ft

    return ParametrosFlujoReservorio(γ, F, ft_val, Leq, D, P0, T0, Rgas)
end

parametros_desde_entrada

### Ejemplo con la interfaz simplificada

Reproducimos el problema de dos tanques de nitrógeno conectados por una tubería de 1/2" Sch 40:

- \(P_0 = 9\ \text{bar(abs)}\),
- \(P_2 = 5\ \text{bar(abs)}\),
- \(T_0 \approx 15^\circ\mathrm{C}\),
- DN = `"1/2"`,
- \(L_e = 18.5\ \mathrm{m}\),
- Gas `"N2"`.

El valor de \(f_T\) se toma automáticamente de la tabla `FT_CRANE`.


In [16]:
# Ejemplo: uso de flujo_masico_reservorios_simple

P0_bar_ej = 9.0
P2_bar_ej = 5.0
T0_C_ej   = 15.0    # °C
DN_ej     = "1/2"
Leq_ej    = 18.5    # m

res_simple = flujo_masico_reservorios_simple(P0_bar_ej, P2_bar_ej, T0_C_ej, DN_ej;
                                             gas = "N2", Leq = Leq_ej)

@show res_simple.r;
@show res_simple.z;
@show res_simple.X;
@show res_simple.mdot;


res_simple.r = 0.9925569433141107
res_simple.z = 0.5607974548067838
res_simple.X = 0.020577506309096497
res_simple.mdot = 0.08653815466710822
